# Load Dataset

In [24]:
pip install ucimlrepo

Note: you may need to restart the kernel to use updated packages.


In [25]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
auto_mpg = fetch_ucirepo(id=9) 
  
# data (as pandas dataframes) 
X = auto_mpg.data.features 
y = auto_mpg.data.targets 

print(X.head())
print(y.head())
  
# metadata 
print(auto_mpg.metadata) 
  
# variable information 
print(auto_mpg.variables) 


   displacement  cylinders  horsepower  weight  acceleration  model_year  \
0         307.0          8       130.0    3504          12.0          70   
1         350.0          8       165.0    3693          11.5          70   
2         318.0          8       150.0    3436          11.0          70   
3         304.0          8       150.0    3433          12.0          70   
4         302.0          8       140.0    3449          10.5          70   

   origin  
0       1  
1       1  
2       1  
3       1  
4       1  
    mpg
0  18.0
1  15.0
2  18.0
3  16.0
4  17.0
{'uci_id': 9, 'name': 'Auto MPG', 'repository_url': 'https://archive.ics.uci.edu/dataset/9/auto+mpg', 'data_url': 'https://archive.ics.uci.edu/static/public/9/data.csv', 'abstract': 'Revised from CMU StatLib library, data concerns city-cycle fuel consumption', 'area': 'Other', 'tasks': ['Regression'], 'characteristics': ['Multivariate'], 'num_instances': 398, 'num_features': 7, 'feature_types': ['Real', 'Categorical', '

# Clean dataset

In [26]:
import pandas as pd
import numpy as np

# Replace ? with NaN
X["horsepower"] = pd.to_numeric(X["horsepower"], errors="coerce")

# Drop rows with missing values
X = X.dropna()

# Align y with cleaned X
y = y.loc[X.index]

# Convert to arrays
X = X.values
y = y.values.flatten()

/var/folders/08/myqmck015r967p6wp3kffhmc0000gn/T/ipykernel_48149/1907059455.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X["horsepower"] = pd.to_numeric(X["horsepower"], errors="coerce")


In [27]:
def normalize(X):
    mean = np.mean(X, axis=0)
    std = np.std(X, axis=0)
    return (X - mean) / std

X = normalize(X)

In [28]:
def add_bias(X):
    ones = np.ones((X.shape[0],1))
    return np.hstack((ones, X))

X = add_bias(X)

# Implement Ridge Regression

In [29]:
def ridge_regression(X, y, lam):

    n_features = X.shape[1]

    I = np.eye(n_features)

    # do not regularize bias
    I[0,0] = 0

    XtX = X.T @ X
    XtY = X.T @ y

    theta = np.linalg.inv(XtX + lam * I) @ XtY

    return theta

In [30]:
# prediection function
def predict(X, theta):
    return X @ theta

In [31]:
# mean squared error
def mse(y_true, y_pred):
    return np.mean((y_true - y_pred)**2)

In [32]:
# 5-fold cross validation
def k_fold_split(n_samples, k=5):

    indices = np.arange(n_samples)
    np.random.shuffle(indices)

    folds = np.array_split(indices, k)

    return folds

# Implement Cross Validation

In [33]:
def cross_validate(X, y, lam, k=5):
    folds = k_fold_split(X.shape[0], k)
    val_errors = []

    for i in range(k):

        # validation set is fold i 
        val_idx = folds[i]

        # training set is all other folds combined
        train_idx = np.concatenate([folds[j] for j in range(k) if j != i])

        X_train, y_train = X[train_idx], y[train_idx]
        X_val, y_val = X[val_idx], y[val_idx]

        theta = ridge_regression(X_train, y_train, lam)
        y_pred = predict(X_val, theta)

        val_errors.append(mse(y_val, y_pred))

    return np.mean(val_errors)

# Grid Search for Optimal Lambda

In [34]:
lambda_grid = [0.0001, 0.001, 0.01, 0.1, 1, 10, 100]
results = {}

for lam in lambda_grid:
    avg_mse = cross_validate(X, y, lam)
    results[lam] = avg_mse
    print(f"lambda={lam:8.4f}  CV MSE={avg_mse:.4f}")

best_lam = min(results, key=results.get)
print(f"\nBest lambda: {best_lam}")

lambda=  0.0001  CV MSE=11.4511
lambda=  0.0010  CV MSE=11.5328
lambda=  0.0100  CV MSE=11.5341
lambda=  0.1000  CV MSE=11.1708
lambda=  1.0000  CV MSE=11.3448
lambda= 10.0000  CV MSE=11.4854
lambda=100.0000  CV MSE=12.8612

Best lambda: 0.1


# Fit Final Model with Best Lambda

In [35]:
final_theta = ridge_regression(X, y, best_lam)
final_pred = predict(X, final_theta)
print(f"mse final training: {mse(y, final_pred):.4f}")

mse final training: 10.8475
